# Przygotowanie danych BUSI do detekcji YOLO

Notebook wykonuje preprocessing:
1. Powielenie obrazow przez losowy szum Gaussa (osobny folder).
2. Konwersja masek do bbox w formacie YOLO.
3. Split train/val/test.
4. Zapis data.yaml dla Ultralytics.

In [1]:
%pip install -q numpy pillow pyyaml scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from dataclasses import dataclass
import random
import shutil

import numpy as np
from PIL import Image
import yaml
from sklearn.model_selection import train_test_split

In [3]:
SOURCE_ROOT = Path('Dataset_BUSI_with_GT')
NOISY_ROOT = Path('Dataset_BUSI_noisy')
YOLO_ROOT = Path('Dataset_BUSI_YOLO')

CLASSES = ['benign', 'malignant', 'normal']
DETECTION_CLASSES = ['benign', 'malignant']
CLASS_TO_ID = {name: idx for idx, name in enumerate(DETECTION_CLASSES)}

COPIES_PER_IMAGE = 1
SIGMA_MIN = 8.0
SIGMA_MAX = 30.0
RANDOM_SEED = 42

TEST_SIZE = 0.10
VAL_SIZE = 0.20

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('SOURCE_ROOT:', SOURCE_ROOT.resolve())
print('NOISY_ROOT: ', NOISY_ROOT.resolve())
print('YOLO_ROOT:  ', YOLO_ROOT.resolve())

SOURCE_ROOT: /Users/kacperciesla/Documents/Studia/Głębokie uczenie maszynowe/neural_networks/Dataset_BUSI_with_GT
NOISY_ROOT:  /Users/kacperciesla/Documents/Studia/Głębokie uczenie maszynowe/neural_networks/Dataset_BUSI_noisy
YOLO_ROOT:   /Users/kacperciesla/Documents/Studia/Głębokie uczenie maszynowe/neural_networks/Dataset_BUSI_YOLO


In [4]:
VALID_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def is_base_image(path: Path) -> bool:
    return path.suffix.lower() in VALID_EXTENSIONS and '_mask' not in path.stem.lower()

def add_gaussian_noise(image_arr: np.ndarray, sigma: float) -> np.ndarray:
    noise = np.random.normal(loc=0.0, scale=sigma, size=image_arr.shape)
    noisy = image_arr.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def find_masks_for_image(image_path: Path) -> list[Path]:
    return sorted(image_path.parent.glob(f'{image_path.stem}_mask*.png'))

def mask_to_bboxes(mask_arr: np.ndarray) -> list[tuple[int, int, int, int]]:
    h, w = mask_arr.shape
    visited = np.zeros((h, w), dtype=bool)
    bboxes = []

    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

    ys, xs = np.where(mask_arr > 0)
    for y0, x0 in zip(ys, xs):
        if visited[y0, x0]:
            continue

        stack = [(y0, x0)]
        visited[y0, x0] = True

        min_y = max_y = y0
        min_x = max_x = x0

        while stack:
            y, x = stack.pop()
            min_y = min(min_y, y)
            max_y = max(max_y, y)
            min_x = min(min_x, x)
            max_x = max(max_x, x)

            for dy, dx in directions:
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w and not visited[ny, nx] and mask_arr[ny, nx] > 0:
                    visited[ny, nx] = True
                    stack.append((ny, nx))

        bboxes.append((min_x, min_y, max_x, max_y))

    return bboxes

def to_yolo_bbox(box, width, height):
    x1, y1, x2, y2 = box
    bw = x2 - x1 + 1
    bh = y2 - y1 + 1
    cx = x1 + bw / 2
    cy = y1 + bh / 2
    return (cx / width, cy / height, bw / width, bh / height)

In [5]:
@dataclass
class Sample:
    image_path: Path
    class_name: str
    bboxes: list[tuple[int, int, int, int]]

def collect_base_samples(source_root: Path) -> list[Sample]:
    samples = []
    for class_name in CLASSES:
        class_dir = source_root / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f'Brak katalogu klasy: {class_dir}')

        for image_path in sorted(class_dir.iterdir()):
            if not image_path.is_file() or not is_base_image(image_path):
                continue

            masks = find_masks_for_image(image_path)
            image_bboxes = []

            for mask_path in masks:
                mask = np.array(Image.open(mask_path).convert('L'))
                image_bboxes.extend(mask_to_bboxes(mask))

            if class_name in CLASS_TO_ID and len(image_bboxes) == 0:
                continue

            samples.append(Sample(image_path=image_path, class_name=class_name, bboxes=image_bboxes))

    return samples

base_samples = collect_base_samples(SOURCE_ROOT)
print('Liczba bazowych probek:', len(base_samples))

Liczba bazowych probek: 780


In [6]:
def build_noisy_dataset(base_samples: list[Sample], noisy_root: Path):
    if noisy_root.exists():
        shutil.rmtree(noisy_root)

    entries = []

    for class_name in CLASSES:
        (noisy_root / class_name).mkdir(parents=True, exist_ok=True)

    for sample in base_samples:
        out_class_dir = noisy_root / sample.class_name
        img = Image.open(sample.image_path).convert('RGB')
        arr = np.array(img)

        original_target = out_class_dir / sample.image_path.name
        img.save(original_target)
        entries.append((original_target, sample.class_name, sample.bboxes))

        for i in range(1, COPIES_PER_IMAGE + 1):
            sigma = random.uniform(SIGMA_MIN, SIGMA_MAX)
            noisy_arr = add_gaussian_noise(arr, sigma)
            noisy_img = Image.fromarray(noisy_arr)

            noisy_name = f'{sample.image_path.stem}__noise_{i:02d}{sample.image_path.suffix.lower()}'
            noisy_target = out_class_dir / noisy_name
            noisy_img.save(noisy_target)
            entries.append((noisy_target, sample.class_name, sample.bboxes))

    return entries

entries = build_noisy_dataset(base_samples, NOISY_ROOT)
print('Liczba plikow po augmentacji:', len(entries))

Liczba plikow po augmentacji: 1560


In [7]:
def split_entries(entries):
    stratify_labels = [cls for _, cls, _ in entries]

    train_val, test = train_test_split(
        entries,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=stratify_labels
    )

    train_val_labels = [cls for _, cls, _ in train_val]
    train, val = train_test_split(
        train_val,
        test_size=VAL_SIZE,
        random_state=RANDOM_SEED,
        stratify=train_val_labels
    )

    return {'train': train, 'val': val, 'test': test}

splits = split_entries(entries)
for split_name, split_entries_list in splits.items():
    print(split_name, len(split_entries_list))

train 1123
val 281
test 156


In [8]:
def prepare_yolo_structure(yolo_root: Path):
    if yolo_root.exists():
        shutil.rmtree(yolo_root)

    for split_name in ['train', 'val', 'test']:
        (yolo_root / 'images' / split_name).mkdir(parents=True, exist_ok=True)
        (yolo_root / 'labels' / split_name).mkdir(parents=True, exist_ok=True)

def write_label_file(label_path: Path, class_name: str, bboxes, width: int, height: int):
    lines = []

    if class_name in CLASS_TO_ID:
        class_id = CLASS_TO_ID[class_name]
        for box in bboxes:
            x, y, w, h = to_yolo_bbox(box, width, height)
            lines.append(f'{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')

    label_path.write_text('\n'.join(lines), encoding='utf-8')

def export_yolo_dataset(splits_dict, yolo_root: Path):
    prepare_yolo_structure(yolo_root)

    for split_name, split_entries_list in splits_dict.items():
        for idx, (img_path, class_name, bboxes) in enumerate(split_entries_list):
            image = Image.open(img_path).convert('RGB')
            width, height = image.size

            target_name = f'{class_name}_{idx:05d}{img_path.suffix.lower()}'
            target_img_path = yolo_root / 'images' / split_name / target_name
            target_lbl_path = yolo_root / 'labels' / split_name / f'{Path(target_name).stem}.txt'

            image.save(target_img_path)
            write_label_file(target_lbl_path, class_name, bboxes, width, height)

    data_yaml = {
        'path': str(yolo_root.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': DETECTION_CLASSES
    }

    (yolo_root / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding='utf-8')

export_yolo_dataset(splits, YOLO_ROOT)
print('Zapisano dataset YOLO do:', YOLO_ROOT.resolve())
print('Plik konfiguracyjny:', (YOLO_ROOT / 'data.yaml').resolve())

Zapisano dataset YOLO do: /Users/kacperciesla/Documents/Studia/Głębokie uczenie maszynowe/neural_networks/Dataset_BUSI_YOLO
Plik konfiguracyjny: /Users/kacperciesla/Documents/Studia/Głębokie uczenie maszynowe/neural_networks/Dataset_BUSI_YOLO/data.yaml


In [9]:
for split_name in ['train', 'val', 'test']:
    n_images = len(list((YOLO_ROOT / 'images' / split_name).glob('*.*')))
    n_labels = len(list((YOLO_ROOT / 'labels' / split_name).glob('*.txt')))
    print(f'{split_name}: images={n_images}, labels={n_labels}')

print('Gotowe. Teraz uruchom detection.ipynb.')

train: images=1123, labels=1123
val: images=281, labels=281
test: images=156, labels=156
Gotowe. Teraz uruchom detection.ipynb.
